# AAMS Week 4 – Continuous-Time Models and AI

Exercise notebook covering all exercises from the Week 4 lecture slides:
- **Part I**: Rabbit/Leslie repetition, Classification table, Ex. 6.1 (first-order form), clock variable, the discrete↔continuous connection
- **Part II**: Euler forward method (Ex. 6.4, 6.5), better integrators (Ex. 6.6), step-size explorer, linear continuous systems
- **Part III**: AI and differential equations – gradient descent as Euler, neuron & forward-mode autodiff, PINN, SINDy
- **Part IV**: Equilibrium points (Ex. 7.1)
- **Part V**: Phase line, basins of attraction, phase space of the pendulum (Ex. 7.4)
- **Part VI**: Variable rescaling of the continuous logistic model

**How it works:** same as in Week 3. Each section has a short theory part, a function with `# TODO: implement`,
and a test cell that fails until your implementation is correct.
Hints are collapsed and graded – open only as many as you need.
Plotting helpers are finished so you can focus on the logic.

Convention as in the lecture: $\dot x = dx/dt = G(x)$ (autonomous) or $f(t, y)$ in the general form. `euler`, `solve_ivp` and all step functions use the signature `f(t, y)`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.linalg import expm
from scipy.optimize import fsolve
from scipy.signal import savgol_filter

plt.rcParams["figure.dpi"] = 110


def simulate_discrete(step, state0, T):
    '''Discrete-time simulator from Week 3: x_t = F(x_{t-1}), returns T+1 states.'''
    state0 = np.asarray(state0, dtype=float)
    ts = np.empty((T + 1,) + state0.shape, dtype=float)
    ts[0] = state0
    for t in range(1, T + 1):
        ts[t] = step(ts[t - 1])
    return ts


def plot_timeseries(t, y, labels=None, title="", ax=None, logy=False):
    y = np.asarray(y, dtype=float)
    if y.ndim == 1:
        y = y[:, None]
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 3.5))
    for i in range(y.shape[1]):
        ax.plot(t, y[:, i], label=labels[i] if labels else None)
    if logy:
        ax.set_yscale("log")
    ax.set_xlabel("t")
    ax.set_title(title)
    ax.grid(alpha=0.3)
    if labels:
        ax.legend()
    return ax


def plot_phase(trajs, labels=("x", "y"), title="Phase space", ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    trajs = trajs if isinstance(trajs, list) else [trajs]
    for tr in trajs:
        tr = np.asarray(tr, dtype=float)
        p = ax.plot(tr[:, 0], tr[:, 1], lw=1, alpha=0.9)
        ax.plot(tr[0, 0], tr[0, 1], "o", color=p[0].get_color(), ms=5)
    ax.set_xlabel(labels[0])
    ax.set_ylabel(labels[1])
    ax.set_title(title)
    ax.grid(alpha=0.3)
    return ax


def plot_vector_field(f, xlim, ylim, ax=None, title="", n=30, labels=("x", "y")):
    '''Streamplot of a 2D system f(t, s) -> [ds0, ds1]. f must work on arrays s = [X, Y].'''
    if ax is None:
        _, ax = plt.subplots(figsize=(5.5, 5))
    X, Y = np.meshgrid(np.linspace(*xlim, n), np.linspace(*ylim, n))
    d = np.array(f(0.0, np.array([X, Y])))
    ax.streamplot(X, Y, d[0], d[1], density=1.3, color="tab:blue", linewidth=0.8)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_xlabel(labels[0])
    ax.set_ylabel(labels[1])
    ax.set_title(title)
    return ax

## 0 · The Continuous-Time Simulator: Forward Euler (Slides 16–17)

Simulating a continuous-time model means numerically integrating it. The simplest method is **forward Euler**:
$$\frac{dy}{dt} = f(t, y) \quad\Rightarrow\quad y_{n+1} = y_n + h\, f(t_n, y_n), \qquad t_n = n\,h$$

This is identical to simulating a discrete-time model, with one technical difference: we have to keep track of **time** as well as the state.

**Contract:** `euler(f, y0, t_end, dt)` returns a tuple `(t, y)` with `t.shape == (n+1,)` and
`y.shape == (n+1,) + np.shape(y0)`, where `n = round(t_end / dt)`. The function `f(t, y)` takes the current time and state and returns the derivative.

In [2]:
def euler(f, y0, t_end, dt):
    '''Forward Euler for dy/dt = f(t, y). Returns (t, y), both including the initial point.'''
    # TODO: implement
    raise NotImplementedError

In [3]:
# Tests
t, y = euler(lambda t, y: y, 1.0, 1.0, 0.5)
assert np.allclose(t, [0, 0.5, 1.0]) and np.allclose(y, [1.0, 1.5, 2.25]), f"got t={t}, y={y}"
assert y.shape == (3,)

t, y = euler(lambda t, s: np.array([s[1], -s[0]]), [1.0, 0.0], 1.0, 0.5)
assert y.shape == (3, 2) and np.allclose(y[1], [1.0, -0.5])

t, y = euler(lambda t, y: t, 0.0, 1.0, 0.5)
assert np.allclose(y, [0.0, 0.0, 0.25]), "f must be evaluated at the OLD time t_n"
print("✓ euler")

NotImplementedError: 

<details><summary>Hint 1</summary>

Create the time grid first (`np.arange(n + 1) * dt`), allocate `y = np.empty((n + 1,) + y0.shape)`, set `y[0] = y0`, then loop over `k in range(n)`.

</details>

<details><summary>Hint 2</summary>

Inside the loop: `y[k + 1] = y[k] + dt * f(t[k], y[k])`. Use the time *at the start* of the step – that is what the third test checks.

</details>

## 1 · Repetition: Age-Structured Rabbit Population (Slides 2–3)

Age classes 0, 1, 2. 50 % of newborns survive their first year, 50 % of those survive the next; the maximum lifespan is two years. Rabbits produce 0, 6 and 8 young in their first, second and third year.

$$\mathbf{x}_{t+1} = L\,\mathbf{x}_t$$

**Tasks:**
1. Implement `leslie_matrix()`.
2. Implement `dominant_eigenpair(L)` returning $\lambda_d$ and the eigenvector scaled so that its entries sum to 1 (the long-run age shares).
3. Implement `age_shares(x)` and check that a simulated population converges to that age structure.

In [ ]:
def leslie_matrix():
    '''Leslie matrix L of the rabbit population (rows: next year's age class 0,1,2).'''
    # TODO: implement
    raise NotImplementedError


def dominant_eigenpair(L):
    '''Returns (lambda_d, v_d): real dominant eigenvalue and its eigenvector scaled to sum 1.'''
    # TODO: implement
    raise NotImplementedError


def age_shares(x):
    '''Share of each age class (sums to 1).'''
    # TODO: implement
    raise NotImplementedError

In [ ]:
# Tests
L = leslie_matrix()
assert np.allclose(L, [[0, 6, 8], [0.5, 0, 0], [0, 0.5, 0]])
lam, v = dominant_eigenpair(L)
assert np.isclose(lam, 2.0), f"Expected lambda_d = 2, got {lam}"
assert np.allclose(v, np.array([16, 4, 1]) / 21), f"Expected (16,4,1)/21, got {v}"
assert np.isclose(v.sum(), 1.0)

ts = simulate_discrete(lambda s: L @ s, [10.0, 0.0, 0.0], 40)
assert np.allclose(age_shares(ts[-1]), v, atol=1e-3), "age structure must converge to the eigenvector"
assert np.isclose(ts[-1].sum() / ts[-2].sum(), 2.0, atol=1e-3), "population doubles every year"
print("✓ Leslie rabbits")
print(f"lambda_d = {lam:.3f}, long-run shares = {np.round(v * 100, 1)} %")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
plot_timeseries(np.arange(41), ts, labels=["age 0", "age 1", "age 2"], title="Population per age class (log)", ax=axes[0], logy=True)
plot_timeseries(np.arange(41), ts / ts.sum(axis=1, keepdims=True), labels=["age 0", "age 1", "age 2"], title="Share of each age class", ax=axes[1])
plt.tight_layout();

<details><summary>Hint 1</summary>

The Leslie matrix has the fertilities in the first row and the survival rates on the sub-diagonal.

</details>

<details><summary>Hint 2</summary>

`w, V = np.linalg.eig(L)`; pick the column of `V` belonging to `np.argmax(np.abs(w))`. Eigenvectors come with an arbitrary scale and sign – dividing by `v.sum()` fixes both.

</details>

<details><summary>Interpretation</summary>

$\lambda^3 - 3\lambda - 2 = (\lambda-2)(\lambda+1)^2$, so $\lambda_1 = 2$ and $\lambda_{2,3} = -1$. Only $|\lambda_1| > 1$, so $\mathbf{x}_t \approx c\cdot 2^t \mathbf{v}_1$: the population doubles every year and the age structure settles on $\approx 76\,\% / 19\,\% / 5\,\%$.

</details>

## 2 · Exercise: Classification of Differential Equations (Slides 7, 10–11)

For each equation decide: **order**, **linear?**, **autonomous?**. (Where the answer depends on an unspecified function such as $M(t)$ or $F(t)$, assume it is a general function of $t$.)

| # | Equation | # | Equation |
|---|---|---|---|
| 1 | $P' = aP(1 - P/b)$ | 6 | $t\,y'' + y' = 0$ |
| 2 | $T' = -k(T - M(t))$ | 7 | $m\,x'' + c\,x' + kx = F(t)$ |
| 3 | $x' = 10 - x/20$ | 8 | $\theta'' + (g/L)\sin\theta = 0$ |
| 4 | $h' = -k\sqrt{h}$ | 9 | $x'' - \mu(1-x^2)x' + x = 0$ |
| 5 | $y' = x/y + y/x$ (independent variable $x$) | 10 | $y'' = (1/a)\sqrt{1 + (y')^2}$ |

**Task:** fill the dictionary `classification` with `(order, is_linear, is_autonomous)` tuples.

In [ ]:
# TODO: fill in (order, is_linear, is_autonomous) for each equation, e.g. 1: (1, False, True)
classification = {
    1: None,
    2: None,
    3: None,
    4: None,
    5: None,
    6: None,
    7: None,
    8: None,
    9: None,
    10: None,
}

In [ ]:
# Tests
expected = {
    1: (1, False, True), 2: (1, True, False), 3: (1, True, True), 4: (1, False, True), 5: (1, False, False),
    6: (2, True, False), 7: (2, True, False), 8: (2, False, True), 9: (2, False, True), 10: (2, False, True),
}
for i, exp in expected.items():
    assert classification[i] is not None, f"Equation {i} is not filled in"
    for name, got, want in zip(["order", "linear", "autonomous"], classification[i], exp):
        assert got == want, f"Equation {i}: wrong {name} (got {got}, expected {want})"
print("✓ classification")

<details><summary>Hint</summary>

- *Linear* means the unknown function and its derivatives appear only to the first power and are never multiplied together or put inside another function. Coefficients may depend on the independent variable.
- *Autonomous* means the independent variable ($t$ or $x$) does not appear explicitly. For #5, remember that $x$ is the **independent** variable.

</details>

<details><summary>Reasons</summary>

1: $P^2$ term → nonlinear. 2: linear in $T$, $t$ enters through $M(t)$ → non-autonomous (autonomous if $M$ is constant). 4: $\sqrt h$ → nonlinear. 5: $1/y$ nonlinear, $x$ explicit. 6: coefficient $t$ → non-autonomous. 7: forcing $F(t)$. 8: $\sin\theta$. 9: $x^2 x'$ product. 10: $y'$ inside a root.

</details>

## 3 · Converting to First-Order Autonomous Form (Slides 7–9, Ex. 6.1)

Every higher-order, non-autonomous ODE can be turned into an autonomous first-order system by adding state variables.

**Exercise 6.1:**
$$\frac{d^2x}{dt^2} - x\frac{dx}{dt} + x^2 = 0$$

**Driven pendulum (non-autonomous):**
$$\ddot\theta = -\frac{g}{L}\sin\theta + k\sin(2\pi f t + \phi)$$
Add the velocity $\omega = \dot\theta$ and a **clock variable** $\dot\tau = 1,\ \tau(0)=0$ to make it autonomous.

**Tasks:**
1. Implement `rhs_61(t, s)` for $s = (x, v)$ with $v = \dot x$.
2. Implement the general converter `to_first_order(g)` for a second-order ODE $\ddot x = g(t, x, \dot x)$.
3. Implement `driven_pendulum_autonomous(s, ...)` for the 3D state $s=(\theta, \omega, \tau)$. It must **not** use an external time argument.

In [ ]:
def rhs_61(t, s):
    '''First-order form of x'' - x x' + x^2 = 0 with s = (x, v), v = x'.'''
    # TODO: implement
    raise NotImplementedError


def to_first_order(g):
    '''Given g(t, x, v) = x'', returns f(t, s) with s = (x, v) so that s' = f(t, s).'''
    # TODO: implement
    raise NotImplementedError


def driven_pendulum_nonautonomous(t, s, g_L=1.0, k=0.5, f=0.3, phi=0.0):
    '''Reference (given): 2D state (theta, omega), explicit time dependence.'''
    return [s[1], -g_L * np.sin(s[0]) + k * np.sin(2 * np.pi * f * t + phi)]


def driven_pendulum_autonomous(s, g_L=1.0, k=0.5, f=0.3, phi=0.0):
    '''Autonomous form with s = (theta, omega, tau). Returns the 3 derivatives.'''
    # TODO: implement
    raise NotImplementedError

In [ ]:
# Tests
assert np.allclose(rhs_61(0.0, [2.0, 3.0]), [3.0, 2.0]), "dx/dt = y, dy/dt = x*y - x^2"

f61 = to_first_order(lambda t, x, v: x * v - x ** 2)
assert np.allclose(f61(0.0, [2.0, 3.0]), rhs_61(0.0, [2.0, 3.0]))
f_pend = to_first_order(lambda t, x, v: -np.sin(x))
assert np.allclose(f_pend(0.0, [0.5, 0.2]), [0.2, -np.sin(0.5)])

# The autonomous system must reproduce the non-autonomous one, as long as tau starts at 0
t, y2 = euler(driven_pendulum_nonautonomous, [0.3, 0.0], 20.0, 0.01)
t, y3 = euler(lambda t, s: driven_pendulum_autonomous(s), [0.3, 0.0, 0.0], 20.0, 0.01)
assert np.allclose(y3[:, :2], y2), "autonomous 3D system must match the driven pendulum"
assert np.allclose(y3[:, 2], t), "the clock variable must satisfy tau(t) = t"
print("✓ first-order conversions")

plot_timeseries(t, y3[:, :2], labels=[r"$\theta$", r"$\omega$"], title="Driven pendulum (autonomous 3D form)");

<details><summary>Hint 1</summary>

Introduce $y = dx/dt$. Then $dx/dt = y$, and the original equation solved for $d^2x/dt^2$ gives $dy/dt$.

</details>

<details><summary>Hint 2</summary>

For the clock: the third component of the derivative is the constant `1.0`. Wherever the non-autonomous version used `t`, use `tau` instead.

</details>

<details><summary>Interpretation</summary>

$\dot x = y,\ \dot y = xy - x^2$ (Slide 13). The clock variable moves the explicit time dependence into the state – the price is one more dimension, the gain is an autonomous system you can analyse with equilibria and phase space.

</details>

## 4 · The Connection: Discrete ↔ Continuous (Slide 14)

Approximating $\frac{dx}{dt} \approx \frac{x(t+\Delta t)-x(t)}{\Delta t} = G(x)$ gives $x(t+\Delta t) \approx x(t) + G(x(t))\,\Delta t$.
So the continuous model $\dot x = G(x)$ corresponds to the discrete map
$$F(x) = x + G(x)\,\Delta t$$
(analogous, but **not** equivalent).

**Task:** implement `euler_map(G, dt)` returning the function $F$, and verify that iterating $F$ equals `euler`.

In [ ]:
def euler_map(G, dt):
    '''Returns the discrete-time map F(x) = x + G(x) * dt.'''
    # TODO: implement
    raise NotImplementedError

In [ ]:
# Tests
F = euler_map(lambda x: 2 * x, 0.1)
assert np.isclose(F(1.0), 1.2)

r_, K_ = 0.8, 10.0
G = lambda x: r_ * x * (1 - x / K_)
dt = 0.25
ts_map = simulate_discrete(euler_map(G, dt), 1.0, 40)
t, y_euler = euler(lambda t, x: G(x), 1.0, 40 * dt, dt)
assert np.allclose(ts_map, y_euler), "iterating the Euler map must equal forward Euler"
print("✓ euler_map")

<details><summary>Hint</summary>

Return a function (closure) `lambda x: ...` that uses `G` and `dt`.

</details>

## 5 · Exercise 6.4: Logistic Growth and the Step Size (Slides 17–19, 22, 27)

$$\frac{dx}{dt} = r\,x\left(1 - \frac{x}{K}\right), \quad x(0)=0.1,\ r=0.2,\ K=1,\ 0 \le t \le 50$$

The exact solution is $x(t) = \dfrac{K}{1 + \left(\frac{K - x_0}{x_0}\right) e^{-rt}}$.

**Tasks:**
1. Implement `logistic_G`, `logistic_exact` and `euler_error(dt, ...)` (maximum absolute deviation of the Euler solution from the exact one).
2. Vary $\Delta t$ and see how the result is affected. Euler is a *first-order* method: halving $\Delta t$ should halve the error.
3. Slide 27: in which picture is the Euler method used? Answer by computing one Euler step on a concave solution.

In [ ]:
def logistic_G(x, r=0.2, K=1.0):
    '''Right-hand side r x (1 - x/K).'''
    # TODO: implement
    raise NotImplementedError


def logistic_exact(t, x0=0.1, r=0.2, K=1.0):
    '''Analytical solution of the logistic ODE.'''
    # TODO: implement
    raise NotImplementedError


def euler_error(dt, x0=0.1, r=0.2, K=1.0, T=50.0):
    '''Maximum absolute difference between the Euler solution and the exact solution on the Euler time grid.'''
    # TODO: implement (use euler, logistic_G and logistic_exact)
    raise NotImplementedError


def euler_one_step_above_exact():
    '''For P' = 1 - P, P(0) = 0 and step h = 1: is the Euler estimate above the true solution? (bool)'''
    # TODO: implement
    raise NotImplementedError

In [ ]:
# Tests
assert np.isclose(logistic_G(0.5, 0.2, 1.0), 0.05)
assert np.isclose(logistic_exact(0.0), 0.1)
assert np.isclose(logistic_exact(1e3), 1.0)

e1, e2, e3 = euler_error(1.0), euler_error(0.1), euler_error(0.01)
assert e1 > e2 > e3, "smaller steps must give smaller errors"
assert 5 < e2 / e3 < 15, f"first-order method: 10x smaller dt -> ~10x smaller error, got ratio {e2 / e3:.1f}"
assert euler_one_step_above_exact() is True, "tangent of a concave solution lies above it"
print("✓ Ex. 6.4 euler_error")

# Large time steps: r*dt = 0.002, 0.2, 1, 2, 2.2
fig, axes = plt.subplots(1, 5, figsize=(17, 3.2), sharey=True)
tt = np.linspace(0, 50, 500)
for ax, dt in zip(axes, [0.01, 1.0, 5.0, 10.0, 11.0]):
    t, y = euler(lambda t, x: logistic_G(x), 0.1, 50.0, dt)
    ax.plot(tt, logistic_exact(tt), "C0", label="exact")
    ax.plot(t, y, "C1.-", ms=3, lw=0.8, label="Euler")
    ax.set_title(f"dt = {dt}  (r·dt = {0.2 * dt:.2f})")
    ax.grid(alpha=0.3)
axes[0].legend()
plt.tight_layout();

<details><summary>Hint 1</summary>

Exact solution: `K / (1 + (K - x0) / x0 * np.exp(-r * t))`. Sanity check: at $t=0$ it returns $x_0$.

</details>

<details><summary>Hint 2</summary>

`euler_error`: call `euler(lambda t, x: logistic_G(x, r, K), x0, T, dt)`, evaluate `logistic_exact` on the returned `t`, take `np.max(np.abs(...))`.

</details>

<details><summary>Hint 3 (Slide 27)</summary>

$P' = 1-P$ has the solution $P(t) = 1 - e^{-t}$. One Euler step uses the slope at the start, $P'(0) = 1$, and follows it in a straight line for $h = 1$. The solution is concave, so the tangent lies *above* it: picture 2 (the arrow overshooting the curve).

</details>

<details><summary>Interpretation</summary>

For $r\Delta t < 1$ Euler follows the curve; for $1 < r\Delta t < 2$ it overshoots $K$ but settles; beyond $r\Delta t = 2$ the fixed point $K$ becomes unstable and numerical oscillations appear that are **not** in the model – a pure discretisation artifact.

</details>

## 6 · Exercise 6.5: Lotka–Volterra with Euler and `solve_ivp` (Slides 23–26)

$$\dot x = a x - b x y, \qquad \dot y = -c y + d x y,\qquad x(0)=y(0)=0.1,\ a=b=c=d=1,\ 0\le t<50$$

The exact solution is periodic: the quantity
$$H(x,y) = d\,x - c\ln x + b\,y - a\ln y$$
is **conserved** along trajectories. A numerical method that makes $H$ drift introduces an artifact. (Check: $\dot H = 0$.)

**Tasks:**
1. Implement `lotka_volterra(t, y, a, b, c, d)`, `simulate_lv_euler(dt)` and `simulate_lv_ivp(method)` using `scipy.integrate.solve_ivp`.
2. Implement `lv_invariant(y, ...)` and `max_invariant_drift(y)`.
3. Compare Euler for smaller $\Delta t$ with `RK45` and `BDF`. What does this say about forward Euler?

In [ ]:
def lotka_volterra(t, y, a=1.0, b=1.0, c=1.0, d=1.0):
    '''Predator-prey right-hand side; y = (prey x, predator y).'''
    # TODO: implement
    raise NotImplementedError


def simulate_lv_euler(dt, T=50.0, y0=(0.1, 0.1)):
    '''Returns (t, y) using the euler function above; y has shape (n+1, 2).'''
    # TODO: implement
    raise NotImplementedError


def simulate_lv_ivp(method="RK45", T=50.0, y0=(0.1, 0.1), n_points=5001):
    '''Returns (t, y) from solve_ivp with rtol=1e-9, atol=1e-12, evaluated at n_points; y has shape (n_points, 2).'''
    # TODO: implement
    raise NotImplementedError


def lv_invariant(y, a=1.0, b=1.0, c=1.0, d=1.0):
    '''H(x, y) = d x - c ln x + b y - a ln y for every row of y (shape (n, 2)).'''
    # TODO: implement
    raise NotImplementedError


def max_invariant_drift(y):
    '''max |H(t) - H(0)| along the trajectory.'''
    # TODO: implement
    raise NotImplementedError

In [ ]:
# Tests
assert np.allclose(lotka_volterra(0.0, [2.0, 3.0]), [-4.0, 3.0])
assert np.allclose(lotka_volterra(0.0, [1.0, 1.0]), [0.0, 0.0]), "(c/d, a/b) = (1, 1) is an equilibrium"

t_e, y_e = simulate_lv_euler(0.01)
assert y_e.shape == (5001, 2) and np.allclose(y_e[0], [0.1, 0.1])
t_r, y_r = simulate_lv_ivp("RK45")
assert y_r.shape == (5001, 2)

drift_rk = max_invariant_drift(y_r)
drift_e1 = max_invariant_drift(y_e)
drift_e2 = max_invariant_drift(simulate_lv_euler(0.001)[1])
print(f"drift of H: RK45 = {drift_rk:.2e}, Euler dt=0.01 = {drift_e1:.2e}, Euler dt=0.001 = {drift_e2:.2e}")
assert drift_rk < 1e-4, "RK45 (tight tolerances) should conserve H"
assert drift_e1 > 10 * drift_rk, "forward Euler drifts clearly"
assert drift_e2 < drift_e1, "smaller dt -> smaller drift"
print("✓ Ex. 6.5 Lotka-Volterra")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_timeseries(t_e, y_e, labels=["prey", "predator"], title="Euler dt = 0.01", ax=axes[0])
plot_timeseries(t_r, y_r, labels=["prey", "predator"], title="solve_ivp RK45", ax=axes[1])
plot_phase([y_e, y_r], labels=("prey", "predator"), title="Phase space (Euler spirals out)", ax=axes[2])
axes[2].legend(["Euler", "", "RK45"])
plt.tight_layout();

<details><summary>Hint 1</summary>

`solve_ivp(fun, t_span, y0, method=..., t_eval=..., rtol=..., atol=...)` returns an object with `.t` and `.y`. Note that `sol.y` has shape `(n_states, n_points)` – transpose it to match the Euler output.

</details>

<details><summary>Hint 2</summary>

`lv_invariant` works column-wise: `y[:, 0]` is the prey, `y[:, 1]` the predator. `max_invariant_drift`: `np.max(np.abs(H - H[0]))`.

</details>

<details><summary>Interpretation</summary>

The true orbits are closed curves. Forward Euler turns them into outward spirals (the amplitude grows without bound as the run gets longer) – this is the same effect as in Week 3 when the 2D map had $|\lambda|>1$. Making $\Delta t$ smaller postpones the problem but never removes it, which motivates the better methods in the next section.

</details>

## 7 · Exercise 6.6: Better Integrators (Slide 24)

Investigate methods that work better than forward Euler. You will implement three one-step methods with the signature `step(f, t, y, h) -> y_next`:

| Method | Formula | Order |
|---|---|---|
| Heun (explicit trapezoid) | $y_{n+1} = y_n + \frac h2 (k_1 + k_2)$, $k_1 = f(t_n, y_n)$, $k_2 = f(t_n + h, y_n + h k_1)$ | 2 |
| Classical RK4 | $y_{n+1} = y_n + \frac h6 (k_1 + 2k_2 + 2k_3 + k_4)$ | 4 |
| Backward Euler (implicit) | $y_{n+1} = y_n + h\,f(t_{n+1}, y_{n+1})$ – solve for $y_{n+1}$ | 1 |

**Tasks:**
1. Implement `heun_step`, `rk4_step` and `backward_euler_step` (use `scipy.optimize.fsolve` for the implicit equation).
2. Implement `estimate_order(step)` – the empirical convergence order from the errors at $h$ and $h/2$ on $\dot y = -y$, $y(0)=1$, $t=1$.
3. Show why implicit methods matter: on the stiff problem $\dot y = -50\,y$ with $h = 0.1$, forward Euler explodes while backward Euler does not.

In [ ]:
def forward_euler_step(f, t, y, h):
    '''Given for reference.'''
    return y + h * np.asarray(f(t, y))


def integrate(step, f, y0, t_end, h):
    '''Given: runs any one-step method on a fixed grid. Returns (t, y).'''
    y0 = np.asarray(y0, dtype=float)
    n = int(round(t_end / h))
    t = np.arange(n + 1) * h
    y = np.empty((n + 1,) + y0.shape)
    y[0] = y0
    for k in range(n):
        y[k + 1] = step(f, t[k], y[k], h)
    return t, y


def heun_step(f, t, y, h):
    # TODO: implement
    raise NotImplementedError


def rk4_step(f, t, y, h):
    # TODO: implement
    raise NotImplementedError


def backward_euler_step(f, t, y, h):
    '''Solve z = y + h f(t + h, z) for z (hint: fsolve on the residual, start from a forward Euler guess).'''
    # TODO: implement
    raise NotImplementedError


def estimate_order(step, h=0.1):
    '''Empirical order: log2(err(h) / err(h/2)) at t = 1 for y' = -y, y(0) = 1.'''
    # TODO: implement
    raise NotImplementedError

In [ ]:
# Tests
f_lin = lambda t, y: -y
y1 = heun_step(f_lin, 0.0, np.array(1.0), 0.1)
assert np.isclose(y1, 1 - 0.1 + 0.005), f"Heun: expected 0.905, got {y1}"
y1 = rk4_step(f_lin, 0.0, np.array(1.0), 0.1)
assert np.isclose(y1, 1 - 0.1 + 0.005 - 0.1 ** 3 / 6 + 0.1 ** 4 / 24), f"RK4: got {y1}"
y1 = backward_euler_step(f_lin, 0.0, np.array(1.0), 0.1)
assert np.isclose(y1, 1 / 1.1, atol=1e-6), f"backward Euler: expected 1/1.1, got {y1}"

orders = {name: estimate_order(s) for name, s in [("forward Euler", forward_euler_step), ("Heun", heun_step),
                                                  ("RK4", rk4_step), ("backward Euler", backward_euler_step)]}
print({k: round(v, 2) for k, v in orders.items()})
assert abs(orders["forward Euler"] - 1) < 0.2 and abs(orders["backward Euler"] - 1) < 0.2
assert abs(orders["Heun"] - 2) < 0.3
assert abs(orders["RK4"] - 4) < 0.3

# Stiff test: y' = -50 y, h = 0.1 (|1 - 50 h| = 4 > 1 -> forward Euler is unstable)
stiff = lambda t, y: -50.0 * y
_, y_fe = integrate(forward_euler_step, stiff, 1.0, 2.0, 0.1)
_, y_be = integrate(backward_euler_step, stiff, 1.0, 2.0, 0.1)
assert abs(y_fe[-1]) > 1e3, "forward Euler must explode"
assert abs(y_be[-1]) < 1e-3, "backward Euler must decay"
print("✓ Ex. 6.6 integrators")

# Lotka-Volterra with the same large step: who keeps the orbit closed?
fig, ax = plt.subplots(figsize=(5.5, 5))
for name, s in [("Euler", forward_euler_step), ("Heun", heun_step), ("RK4", rk4_step)]:
    _, y = integrate(s, lotka_volterra, [0.5, 0.5], 30.0, 0.05)
    ax.plot(y[:, 0], y[:, 1], lw=1, label=name)
ax.set_xlabel("prey"); ax.set_ylabel("predator"); ax.set_title("Lotka-Volterra, h = 0.05"); ax.legend(); ax.grid(alpha=0.3);

<details><summary>Hint 1</summary>

All three steps take arrays: wrap `f(...)` in `np.asarray` so that `y + h * k1` works for scalars and vectors alike.

</details>

<details><summary>Hint 2 (backward Euler)</summary>

Define the residual $g(z) = z - y - h f(t+h, z)$; the next state is its root. `fsolve` expects a 1D array, so flatten with `.ravel()` and `.reshape(y.shape)` back at the end.

</details>

<details><summary>Hint 3 (order)</summary>

Run `integrate(step, f, 1.0, 1.0, hh)` for `hh = h` and `h/2`, compare `y[-1]` with `np.exp(-1)`, and compute `np.log2(err_h / err_h2)`.

</details>

<details><summary>Interpretation</summary>

Higher-order methods evaluate the slope more than once per step and therefore track the curvature of the solution: the error shrinks as $h^p$. Implicit methods solve an equation per step but remain stable for large $h$ – the price you pay is the solve, the gain is robustness on stiff problems. `solve_ivp` uses adaptive step sizes on top of this (`RK45`, `DOP853`, `Radau`, `BDF`, `LSODA`, Slide 26).

</details>

## 8 · Step-Size Explorer: Euler Meets the Logistic Map (Slides 18–19, 28)

Forward Euler on logistic growth with step $\Delta t$ is the discrete map
$$x_{n+1} = x_n + r\Delta t\,x_n\left(1 - \frac{x_n}{K}\right).$$
Rescaling with $\alpha = K(1+r\Delta t)/(r\Delta t)$ turns it into the logistic map $x'_{n+1} = r'\,x'_n(1-x'_n)$ with $r' = 1 + r\Delta t$.
All behaviour is controlled by the single number $r\Delta t$:

| $r\Delta t$ | behaviour |
|---|---|
| $<1$ | smooth |
| $1\ldots2$ | overshoot, settles |
| $2\ldots2.5$ | two values (period doubling) |
| $2.5\ldots3$ | chaos |
| $>3$ | blow-up |

**Tasks:**
1. Implement `euler_to_logistic_map(r, dt, K)` returning $(\alpha, r')$.
2. Implement `regime(r_dt)` and `logistic_map_orbit(r_prime, ...)`.
3. Implement `attractor_size(orbit)` and use it to draw the bifurcation diagram.

This is the *program that explores different step sizes* asked for on Slide 28.

In [ ]:
def euler_to_logistic_map(r, dt, K):
    '''Returns (alpha, r_prime) so that x = alpha * x' maps Euler-logistic to x' -> r' x' (1 - x').'''
    # TODO: implement
    raise NotImplementedError


def regime(r_dt):
    '''Returns "smooth" (<1), "overshoot, settles" (<2), "2 values" (<2.5), "chaos" (<=3) or "blow-up" (>3).'''
    # TODO: implement
    raise NotImplementedError


def logistic_map_orbit(r_prime, n_transient=500, n_keep=200, x0=0.1):
    '''Iterate x -> r' x (1 - x); discard n_transient steps, return the next n_keep values.'''
    # TODO: implement
    raise NotImplementedError


def attractor_size(orbit, decimals=4):
    '''Number of distinct values in the orbit (rounded to `decimals`).'''
    # TODO: implement
    raise NotImplementedError

In [ ]:
# Tests
alpha, rp = euler_to_logistic_map(1.2, 1.5, 100.0)
assert np.isclose(rp, 2.8) and np.isclose(alpha, 100.0 * 2.8 / 1.8)

# The Euler-logistic trajectory must equal alpha * (logistic map trajectory)
t, y = euler(lambda t, x: logistic_G(x, 1.2, 100.0), 5.0, 30.0, 1.5)
ts_map = simulate_discrete(lambda xp: rp * xp * (1 - xp), 5.0 / alpha, 20)
assert np.allclose(alpha * ts_map, y), "rescaled logistic map must reproduce the Euler trajectory"

for r_dt, name in [(0.5, "smooth"), (1.5, "overshoot, settles"), (2.2, "2 values"), (2.8, "chaos"), (3.5, "blow-up")]:
    assert regime(r_dt) == name, f"regime({r_dt}) should be '{name}', got '{regime(r_dt)}'"

assert attractor_size(logistic_map_orbit(1 + 0.5)) == 1
assert attractor_size(logistic_map_orbit(1 + 1.5)) == 1
assert attractor_size(logistic_map_orbit(1 + 2.2)) == 2
assert attractor_size(logistic_map_orbit(1 + 2.5)) == 4
assert attractor_size(logistic_map_orbit(1 + 2.8)) > 20
print("✓ step-size explorer")

# Bifurcation diagram, exactly as on the slides (x/K versus r*dt)
fig, ax = plt.subplots(figsize=(7, 4))
for r_dt in np.linspace(0.01, 3.0, 700):
    orbit = logistic_map_orbit(1 + r_dt, 300, 100)
    ax.plot(np.full_like(orbit, r_dt), orbit * (1 + r_dt) / r_dt, ",k")  # x/K = x' * alpha / K = x' * (1 + r_dt) / r_dt
ax.set_xlabel(r"$r\Delta t$"); ax.set_ylabel("x / K"); ax.set_title("Bifurcation diagram of Euler-logistic"); ax.grid(alpha=0.3);

<details><summary>Hint 1</summary>

Replace $r$ by $r\Delta t$ in the Week 3 rescaling: $\alpha = K(1+r\Delta t)/(r\Delta t)$ and $r' = 1 + r\Delta t$.

</details>

<details><summary>Hint 2</summary>

`regime`: chain `if`s on the thresholds 1, 2, 2.5 and 3. `attractor_size`: `np.unique(np.round(orbit, decimals)).size` – rounding collapses numerical noise around a converged cycle.

</details>

<details><summary>Interpretation</summary>

$r' = 3$ (i.e. $r\Delta t = 2$) is the first period doubling of the logistic map: the fixed point loses stability exactly when $|1 - r\Delta t| = 1$. This is the "stable only for $r\Delta t < 2$" rule from the slides. The continuous model never oscillates: everything above $r\Delta t = 2$ is a numerical artifact (Slide 19: "always check whether numerical results are consistent with expected model behavior").

</details>

## 9 · Continuous-Time Linear Systems (Slides 20–21, Takeaway 3)

$$\frac{d\mathbf{x}}{dt} = A\mathbf{x} \quad\Rightarrow\quad \mathbf{x}(t) = e^{At}\mathbf{x}(0)$$

The **eigenvalues** decide the long-term behaviour. In continuous time the criterion is the **real part** (not the absolute value as in discrete time!):

| Eigenvalues of $A$ (2D) | Type |
|---|---|
| real, both $<0$ / both $>0$ | stable / unstable node |
| real, opposite signs | saddle |
| complex, $\mathrm{Re}<0$ / $>0$ | stable / unstable spiral |
| purely imaginary | center |

**Tasks:**
1. Implement `solve_linear(A, x0, t)` with the matrix exponential (`scipy.linalg.expm`).
2. Implement `classify_equilibrium_2d(A)`.
3. Compare with forward Euler: why does Euler spiral outward for a center?

In [ ]:
def solve_linear(A, x0, t):
    '''Returns x(t_k) = expm(A t_k) x0 for every t_k; shape (len(t), n).'''
    # TODO: implement
    raise NotImplementedError


def classify_equilibrium_2d(A, atol=1e-9):
    '''Returns one of: "stable node", "unstable node", "saddle", "stable spiral", "unstable spiral", "center", "degenerate".'''
    # TODO: implement
    raise NotImplementedError

In [ ]:
# Tests
cases = {
    "stable node": [[-1, 0], [0, -2]],
    "unstable node": [[1, 0], [0, 2]],
    "saddle": [[1, 0], [0, -1]],
    "stable spiral": [[-0.1, 1], [-1, -0.1]],
    "unstable spiral": [[0.1, 1], [-1, 0.1]],
    "center": [[0, 1], [-1, 0]],
}
for name, A in cases.items():
    assert classify_equilibrium_2d(np.array(A, float)) == name, f"{A} should be a {name}, got {classify_equilibrium_2d(np.array(A, float))}"

# Rotation: x(t) = (cos t, -sin t)
tt = np.linspace(0, 6, 13)
assert np.allclose(solve_linear([[0, 1], [-1, 0]], [1, 0], tt), np.column_stack([np.cos(tt), -np.sin(tt)]))
# Non-diagonalizable matrix -> polynomial growth in t (footnote on Slide 21)
assert np.allclose(solve_linear([[0, 1], [0, 0]], [0, 1], [0, 1, 2]), [[0, 1], [1, 1], [2, 1]])

# Euler on a center spirals outward: |x|^2 is multiplied by (1 + dt^2) every step
A_c = np.array([[0.0, 1.0], [-1.0, 0.0]])
t, y_eul = euler(lambda t, s: A_c @ s, [1.0, 0.0], 20.0, 0.1)
assert np.linalg.norm(y_eul[-1]) > 2.0, "forward Euler turns a center into an unstable spiral"
t, y_fine = euler(lambda t, s: A_c @ s, [1.0, 0.0], 2.0, 1e-4)
assert np.allclose(y_fine[-1], solve_linear(A_c, [1, 0], [2.0])[0], atol=1e-3)
print("✓ continuous linear systems")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, (name, A) in zip(axes, [("stable spiral", cases["stable spiral"]), ("saddle", cases["saddle"]), ("center", cases["center"])]):
    A = np.array(A, float)
    plot_vector_field(lambda t, s, A=A: [A[0, 0] * s[0] + A[0, 1] * s[1], A[1, 0] * s[0] + A[1, 1] * s[1]], (-2, 2), (-2, 2), ax=ax, title=name)
plt.tight_layout();

<details><summary>Hint 1</summary>

`expm(A * tk) @ x0` for every time `tk` – a list comprehension wrapped in `np.array(...)` gives the right shape.

</details>

<details><summary>Hint 2</summary>

Use `np.linalg.eigvals(A)`. First check whether any imaginary part is non-negligible (complex pair), then look at the sign of the real part. For real eigenvalues, compare the signs of both.

</details>

<details><summary>Interpretation</summary>

An Euler step multiplies the state by $I + hA$, whose eigenvalues are $1 + h\lambda$. For $\lambda = \pm i$ this gives $|1 + ih| = \sqrt{1+h^2} > 1$: a center becomes an unstable spiral. Stability of the *numerical* scheme requires $|1 + h\lambda| < 1$, i.e. $h\lambda$ inside a disk around $-1$ – this is where the rule $r\Delta t<2$ comes from.

</details>

## 10 · Differential Equations for Understanding AI: Gradient Descent = Euler (Slides 47–49)

Gradient flow: $\dot\theta = -\nabla L(\theta)$. A forward-Euler step with $h = \eta$ is exactly gradient descent:
$$\theta_{k+1} = \theta_k - \eta\,\nabla L(\theta_k)$$

On a valley $L(\theta)=\tfrac12\lambda\theta^2$ each step multiplies $\theta$ by $(1-\eta\lambda)$, so it converges iff $\eta\lambda < 2$.
For several parameters the steepest curvature decides: $\eta < 2/\lambda_{\max}(H)$.

**Tasks:**
1. Implement `gd_step`, `gd_quadratic`, `gradient_flow_quadratic`.
2. Implement `gd_converges(eta, lam)` and `max_stable_lr(H)`.

In [ ]:
def gd_step(theta, grad_L, eta):
    '''One gradient descent step: theta - eta * grad_L(theta).'''
    # TODO: implement
    raise NotImplementedError


def gd_quadratic(theta0, lam, eta, n):
    '''n gradient descent steps on L = 0.5 * lam * theta^2. Returns n+1 values (use gd_step).'''
    # TODO: implement
    raise NotImplementedError


def gradient_flow_quadratic(theta0, lam, t):
    '''Exact gradient flow of L = 0.5 * lam * theta^2 at times t.'''
    # TODO: implement
    raise NotImplementedError


def gd_converges(eta, lam):
    '''True iff gradient descent on 0.5 * lam * theta^2 converges for learning rate eta.'''
    # TODO: implement
    raise NotImplementedError


def max_stable_lr(H):
    '''Largest stable learning rate for a quadratic loss with symmetric Hessian H.'''
    # TODO: implement
    raise NotImplementedError

In [ ]:
# Tests
assert np.isclose(gd_step(2.0, lambda th: 3 * th, 0.1), 1.4)
traj = gd_quadratic(2.5, 2.0, 0.1, 5)
assert np.allclose(traj, 2.5 * (1 - 0.2) ** np.arange(6)), "each step multiplies theta by (1 - eta*lam)"
assert np.isclose(gradient_flow_quadratic(2.5, 2.0, 1.0), 2.5 * np.exp(-2.0))

# Small eta follows the gradient flow closely (Euler consistency)
k = np.arange(11)
assert np.allclose(gd_quadratic(2.5, 2.0, 0.001, 10), gradient_flow_quadratic(2.5, 2.0, k * 0.001), atol=1e-4)

assert [gd_converges(e, 2.0) for e in (0.1, 0.3, 0.75, 1.05)] == [True, True, True, False]  # eta*lam = 0.2, 0.6, 1.5, 2.1
assert np.isclose(max_stable_lr(np.diag([1.0, 50.0])), 0.04)
print("✓ gradient descent = Euler")

fig, axes = plt.subplots(1, 4, figsize=(16, 3.2), sharey=True)
for ax, eta in zip(axes, [0.1, 0.3, 0.75, 1.05]):
    n = 12
    ax.plot(np.arange(n + 1) * eta, gd_quadratic(2.5, 2.0, eta, n), "C1o-", ms=3, label="gradient descent")
    tt = np.linspace(0, n * eta, 200)
    ax.plot(tt, gradient_flow_quadratic(2.5, 2.0, tt), "C0", label="gradient flow")
    ax.set_title(rf"$\eta\lambda$ = {eta * 2:.1f}"); ax.set_xlabel(r"$t = k\eta$"); ax.grid(alpha=0.3)
axes[0].legend(); axes[0].set_ylim(-4, 4)
plt.tight_layout();

<details><summary>Hint 1</summary>

$\nabla L(\theta) = \lambda\theta$ for $L = \tfrac12\lambda\theta^2$. The gradient flow solves $\dot\theta = -\lambda\theta$, i.e. $\theta_0 e^{-\lambda t}$.

</details>

<details><summary>Hint 2</summary>

Convergence ⇔ the factor $1-\eta\lambda$ lies strictly between $-1$ and $1$. For a matrix $H$ use `np.linalg.eigvalsh(H)` and the largest eigenvalue.

</details>

<details><summary>Interpretation</summary>

$\eta\lambda \in (0,1)$: monotone approach; $\eta\lambda \in (1,2)$: the iterate overshoots and zigzags across the minimum but still converges; $\eta\lambda>2$: it diverges. In a narrow valley the *steep* direction limits $\eta$, so the shallow direction crawls – exactly the zigzag-across / crawl-along picture on Slide 48. Same rule, same maths as $r\Delta t<2$ for logistic Euler.

</details>

## 11 · Neurons, Layers and Automatic Differentiation (Slides 30–33, 37)

**Neuron:** $y = \sigma(w_1x_1 + \dots + w_nx_n + b)$. Layers feed forward; a hidden layer uses a non-linear activation, the output layer here is linear.

**Forward-mode automatic differentiation:** every value carries its derivative along (chain rule per operation).
For $f(x) = x\sin x$ at $x=2$: $v_1=x$, $v_1'=1$; $v_2=\sin v_1$, $v_2' = \cos v_1\cdot v_1'$; $v_3 = v_1 v_2$, $v_3' = v_1'v_2 + v_1v_2'$. Result: $f'(2) = \sin 2 + 2\cos 2 \approx 0.077$.

**Tasks:**
1. Implement `sigmoid`, `relu`, `neuron` and `mlp_forward`.
2. Implement the `Dual` number class (`+`, `*`) and `dsin`, `dexp`.

In [ ]:
def sigmoid(z):
    # TODO: implement
    raise NotImplementedError


def relu(z):
    # TODO: implement
    raise NotImplementedError


def neuron(x, w, b, act):
    '''y = act(w . x + b)'''
    # TODO: implement
    raise NotImplementedError


def mlp_forward(x, layers, act):
    '''layers = [(W1, b1), ..., (Wk, bk)]. Hidden layers apply act, the last layer is linear.'''
    # TODO: implement
    raise NotImplementedError


class Dual:
    '''Forward-mode AD: carries a value and its derivative with respect to the input x.'''

    def __init__(self, val, der=0.0):
        self.val = val
        self.der = der

    @staticmethod
    def lift(other):
        return other if isinstance(other, Dual) else Dual(other, 0.0)

    def __add__(self, other):
        # TODO: implement (sum rule)
        raise NotImplementedError

    __radd__ = __add__

    def __mul__(self, other):
        # TODO: implement (product rule)
        raise NotImplementedError

    __rmul__ = __mul__


def dsin(u):
    # TODO: implement (chain rule)
    raise NotImplementedError


def dexp(u):
    # TODO: implement (chain rule)
    raise NotImplementedError


def forward_derivative(f, x):
    '''Given: evaluates f on Dual(x, 1) and returns (f(x), f'(x)).'''
    out = Dual.lift(f(Dual(x, 1.0)))
    return out.val, out.der

In [ ]:
# Tests: neurons
assert np.isclose(sigmoid(0.0), 0.5) and sigmoid(20.0) > 0.999999
assert np.allclose(relu(np.array([-1.0, 0.0, 2.0])), [0.0, 0.0, 2.0])
assert np.isclose(neuron([1.0, 2.0], [0.5, -0.25], 0.1, np.tanh), np.tanh(0.5 - 0.5 + 0.1))
layers = [(np.array([[1.0, -1.0], [0.5, 0.5]]), np.zeros(2)), (np.array([[2.0, 1.0]]), np.array([0.5]))]
assert np.allclose(mlp_forward([1.0, 2.0], layers, relu), [2.0]), "hidden = relu([-1, 1.5]) = [0, 1.5]; out = 2*0 + 1*1.5 + 0.5"

# Tests: automatic differentiation (Slide 37)
val, der = forward_derivative(lambda x: x * dsin(x), 2.0)
assert np.isclose(val, 2 * np.sin(2.0)) and np.isclose(der, np.sin(2.0) + 2 * np.cos(2.0)), f"got {val}, {der}"
assert np.isclose(der, 0.077, atol=1e-3)
val, der = forward_derivative(lambda x: 3 * dexp(x * x), 0.5)
assert np.isclose(der, 6 * 0.5 * np.exp(0.25)), f"d/dx 3 exp(x^2) at 0.5, got {der}"
h = 1e-6
fd = (forward_derivative(lambda x: x * dsin(x) * x, 1.3 + h)[0] - forward_derivative(lambda x: x * dsin(x) * x, 1.3 - h)[0]) / (2 * h)
assert np.isclose(forward_derivative(lambda x: x * dsin(x) * x, 1.3)[1], fd, atol=1e-6), "must agree with finite differences"
print("✓ neuron, MLP, forward-mode autodiff")

<details><summary>Hint 1 (MLP)</summary>

Loop over all layers except the last: `h = act(W @ h + b)`. Apply the last layer without activation.

</details>

<details><summary>Hint 2 (Dual)</summary>

Convert `other` with `Dual.lift` first, so that `3 * dexp(...)` and `x + 1.0` work. Sum rule: derivatives add. Product rule: `d = self.der * o.val + self.val * o.der`.

</details>

<details><summary>Hint 3 (dsin)</summary>

Chain rule: value `np.sin(u.val)`, derivative `np.cos(u.val) * u.der`.

</details>

<details><summary>Interpretation</summary>

Automatic differentiation is exact to machine precision (no truncation error like finite differences, no expression swell like symbolic differentiation). Reverse mode is what backpropagation does for $\partial L/\partial\theta$; forward mode gives $\partial u/\partial t$ inside the PINN residual.

</details>

## 12 · Physics-Informed Neural Networks (Slides 34–39)

A PINN $u_\theta(t)$ is trained to fit data **and** to make the residual of the ODE vanish:
$$L(\theta,\lambda) = L_{\text{data}} + \alpha\,L_{\text{physics}} + \beta\,L_{\text{IC}}$$
For logistic growth $\dot u = r u (1-u/K)$ the physics residual is $\dot u - r u(1 - u/K)$, evaluated at **collocation points** (no data needed there).

- **Forward problem:** known $r, K, u_0$ → find $u(t)$.
- **Inverse problem:** known equation form + noisy data → find $u(t)$ **and** $r$ (make `r` a trainable parameter).

This section needs **PyTorch** (`pip install torch`). The tests are skipped if it is not installed. The training loop is given – you implement the three loss terms.

**Tasks:** implement `physics_loss`, `ic_loss`, `data_loss` using `torch.autograd.grad(..., create_graph=True)` for $du/dt$.

In [ ]:
try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
    print("PyTorch not installed - the PINN section is skipped (pip install torch)")


def physics_loss(net, t, r, K):
    '''Mean squared residual of du/dt - r u (1 - u/K) on collocation points t (tensor (N,1), requires_grad=True).'''
    # TODO: implement (autograd for du/dt, create_graph=True)
    raise NotImplementedError


def ic_loss(net, u0):
    '''(u(0) - u0)^2 for the network output at t = 0.'''
    # TODO: implement
    raise NotImplementedError


def data_loss(net, t_data, u_data):
    '''Mean squared error between the network and the measurements.'''
    # TODO: implement
    raise NotImplementedError

In [ ]:
# Tests
if HAS_TORCH:
    r_, K_, u0_ = 1.0, 1.0, 0.1
    exact = lambda t: K_ / (1 + (K_ / u0_ - 1) * torch.exp(-r_ * t))
    t_col = torch.linspace(0, 10, 100).reshape(-1, 1).requires_grad_(True)
    assert physics_loss(exact, t_col, r_, K_) < 1e-10, "the exact solution has zero physics residual"
    assert ic_loss(exact, u0_) < 1e-12
    wrong = lambda t: 0.5 * t
    assert physics_loss(wrong, t_col, r_, K_) > 1e-2, "a wrong function has a large residual"
    assert ic_loss(lambda t: t + 1.0, u0_) > 0.5
    t_d = torch.linspace(0, 10, 11).reshape(-1, 1)
    assert data_loss(exact, t_d, exact(t_d)) < 1e-12
    assert np.isclose(data_loss(lambda t: exact(t) + 0.1, t_d, exact(t_d)).item(), 0.01, atol=1e-6)
    print("✓ PINN loss terms")
else:
    print("skipped (no PyTorch)")

In [ ]:
# Given: training loop (forward and inverse problem). Runs in ~10-30 s on a CPU.
RUN_PINN_DEMO = True

if HAS_TORCH and RUN_PINN_DEMO:
    def make_net(width=32):
        return torch.nn.Sequential(torch.nn.Linear(1, width), torch.nn.Tanh(), torch.nn.Linear(width, width),
                                   torch.nn.Tanh(), torch.nn.Linear(width, 1))

    def train(net, epochs, loss_fn, params=None, lr=1e-2):
        opt = torch.optim.Adam(list(net.parameters()) + (params or []), lr=lr)
        for _ in range(epochs):
            opt.zero_grad()
            loss = loss_fn()
            loss.backward()
            opt.step()
        return loss.item()

    torch.manual_seed(0)
    r_true, K_true, u0_true = 1.0, 1.0, 0.1   # scaled problem: variables and time of order 1 (Slide 39)
    t_col = torch.linspace(0, 10, 100).reshape(-1, 1).requires_grad_(True)
    tt = np.linspace(0, 10, 200)
    u_exact = logistic_exact(tt, u0_true, r_true, K_true)

    # Forward problem: no data, only equation + initial condition
    net_f = make_net()
    train(net_f, 3000, lambda: physics_loss(net_f, t_col, r_true, K_true) + ic_loss(net_f, u0_true))

    # Inverse problem: noisy data + unknown r
    rng = np.random.default_rng(0)
    t_data = torch.linspace(0, 10, 15).reshape(-1, 1)
    u_data = torch.tensor(logistic_exact(t_data.numpy(), u0_true, r_true, K_true) + 0.02 * rng.standard_normal((15, 1)), dtype=torch.float32)
    r_hat = torch.nn.Parameter(torch.tensor(0.3))
    net_i = make_net()
    train(net_i, 4000, lambda: data_loss(net_i, t_data, u_data) + 0.5 * physics_loss(net_i, t_col, r_hat, K_true) + ic_loss(net_i, u0_true), params=[r_hat])

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    axes[0].plot(tt, u_exact, "k", label="exact"); axes[0].plot(tt, net_f(torch.tensor(tt, dtype=torch.float32).reshape(-1, 1)).detach().numpy(), "C1--", label="PINN")
    axes[0].set_title("Forward problem (no data)"); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(tt, u_exact, "k", label="exact"); axes[1].plot(t_data.numpy(), u_data.numpy(), "C3o", label="noisy data")
    axes[1].plot(tt, net_i(torch.tensor(tt, dtype=torch.float32).reshape(-1, 1)).detach().numpy(), "C1--", label="PINN")
    axes[1].set_title(f"Inverse problem: r recovered = {r_hat.item():.2f} (true {r_true})"); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()

<details><summary>Hint 1</summary>

`u = net(t)`; the derivative with respect to the input: `torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True)[0]`. `create_graph=True` is essential – otherwise the residual cannot be differentiated again with respect to the weights.

</details>

<details><summary>Hint 2</summary>

Residual: `du - r * u * (1 - u / K)`; the loss is the mean of its square. For the IC loss evaluate the network at `torch.zeros(1, 1)`.

</details>

<details><summary>Interpretation / practical notes (Slide 39)</summary>

Keep variables and time of order 1 (non-dimensionalise, Section 16!). Balance the loss weights and watch each term separately. Collocation points should cover the whole domain, including where you have no data. Beware of easy wrong answers: $u\equiv0$ or $u\equiv K$ also satisfy the equation – the initial condition or data rules them out. Inverse problems are often **ill-posed** (Slide 38): noise gets amplified and different parameters can fit the same data, so extra assumptions (physics loss, priors) are needed.

</details>

## 13 · SINDy: Discovering Equations from Data (Slides 40–46)

**Sparse Identification of Nonlinear Dynamics** turns model discovery into a sparse linear regression:
$$\dot X \approx \Theta(X)\,\Xi$$
1. **Estimate** $\dot X$ from the measurements (finite differences, after light smoothing).
2. **Build a library** $\Theta(X)$ of candidate terms, here $[1,\ x,\ y,\ x^2,\ xy,\ y^2]$.
3. **Sparse regression** with *sequentially thresholded least squares* (STLSQ): least squares → set $|\xi_j|<\text{threshold}$ to zero → refit the remaining terms → repeat until nothing changes.

Test data (true model, noisy in the last part): $\dot x = 1.0\,x - 0.5\,xy$, $\dot y = -0.75\,y + 0.25\,xy$.

**Tasks:** implement `finite_diff_derivative`, `build_library` and `stlsq`.

In [ ]:
LIBRARY_NAMES = ["1", "x", "y", "x^2", "xy", "y^2"]


def generate_lv_data(noise=0.0, seed=0, dt=0.1, T=20.0):
    '''Given: predator-prey data (a=1, b=0.5, c=0.75, d=0.25, x0=5, y0=1) with relative Gaussian noise.'''
    f = lambda t, s: [1.0 * s[0] - 0.5 * s[0] * s[1], -0.75 * s[1] + 0.25 * s[0] * s[1]]
    t = np.arange(0, T + dt / 2, dt)
    X = solve_ivp(f, [0, T], [5.0, 1.0], t_eval=t, rtol=1e-10, atol=1e-12).y.T
    rng = np.random.default_rng(seed)
    return t, X + noise * X.std(axis=0) * rng.standard_normal(X.shape)


def smoothed_derivative(X, dt, window=15, poly=3):
    '''Given: Savitzky-Golay smoothing + differentiation (the "light smoothing" of Slide 41).'''
    return savgol_filter(X, window, poly, deriv=1, delta=dt, axis=0)


def finite_diff_derivative(X, dt):
    '''Central finite differences along axis 0 (one-sided at the ends); same shape as X.'''
    # TODO: implement
    raise NotImplementedError


def build_library(X):
    '''Library matrix Theta with columns [1, x, y, x^2, x*y, y^2] for X of shape (n, 2).'''
    # TODO: implement
    raise NotImplementedError


def stlsq(Theta, dX, threshold, max_iter=20):
    '''Sequentially thresholded least squares. Returns Xi of shape (n_terms, n_states).'''
    # TODO: implement
    raise NotImplementedError


def format_equations(Xi, names=("x'", "y'")):
    '''Given: pretty-print the discovered equations.'''
    lines = []
    for k, nm in enumerate(names):
        terms = [f"{Xi[j, k]:+.3f} {LIBRARY_NAMES[j]}" for j in range(Xi.shape[0]) if Xi[j, k] != 0]
        lines.append(f"{nm} = " + (" ".join(terms) if terms else "0"))
    return "\n".join(lines)

In [ ]:
# Tests
t, X = generate_lv_data(noise=0.0)
dt = t[1] - t[0]
dX = finite_diff_derivative(X, dt)
assert dX.shape == X.shape
assert np.allclose(finite_diff_derivative(np.array([[0.0], [1.0], [4.0], [9.0]]), 1.0).ravel(), [1.0, 2.0, 4.0, 5.0]), "np.gradient style: central inside, one-sided at the ends"

Theta = build_library(X)
assert Theta.shape == (len(t), 6) and np.allclose(Theta[:, 0], 1) and np.allclose(Theta[:, 4], X[:, 0] * X[:, 1])

true_Xi = np.zeros((6, 2))
true_Xi[1, 0], true_Xi[4, 0], true_Xi[2, 1], true_Xi[4, 1] = 1.0, -0.5, -0.75, 0.25

Xi = stlsq(Theta, dX, threshold=0.05)
assert Xi.shape == (6, 2)
assert np.array_equal(Xi != 0, true_Xi != 0), f"wrong support:\n{format_equations(Xi)}"
assert np.allclose(Xi, true_Xi, atol=0.05), f"coefficients off:\n{format_equations(Xi)}"

# Threshold too strict: d = 0.25 falls below 0.3 -> the y equation collapses (Slide 42)
Xi_strict = stlsq(Theta, dX, threshold=0.3)
assert Xi_strict[4, 1] == 0.0 and Xi_strict[2, 1] == 0.0, "y-equation should collapse for threshold 0.3"

# Noisy data (2 %), smoothed derivative (Slide 43)
t, Xn = generate_lv_data(noise=0.02, seed=1)
Xi_n = stlsq(build_library(Xn), smoothed_derivative(Xn, dt), threshold=0.05)
assert np.array_equal(Xi_n != 0, true_Xi != 0), f"noisy support wrong:\n{format_equations(Xi_n)}"
print("✓ SINDy")
print("\nDiscovered from 2 % noisy data:\n" + format_equations(Xi_n))

# Noise study (Slide 44): success rate of finding exactly the true terms
for noise in [0.02, 0.05, 0.10, 0.15]:
    ok_fd = ok_sm = 0
    for seed in range(10):
        _, Xs = generate_lv_data(noise, seed)
        for name, d in [("fd", finite_diff_derivative(Xs, dt)), ("sm", smoothed_derivative(Xs, dt))]:
            hit = np.array_equal(stlsq(build_library(Xs), d, 0.05) != 0, true_Xi != 0)
            ok_fd += hit and name == "fd"; ok_sm += hit and name == "sm"
    print(f"noise {noise:4.0%}: finite differences {ok_fd}/10, smoothed {ok_sm}/10")

<details><summary>Hint 1</summary>

`np.gradient(X, dt, axis=0)` already does central differences with one-sided ends.

</details>

<details><summary>Hint 2 (STLSQ)</summary>

Start with `np.linalg.lstsq(Theta, dX, rcond=None)[0]`. In the loop: build a boolean mask `small = np.abs(Xi) < threshold`, set those entries to 0, then refit **each state column separately** using only the columns of `Theta` that are still active (`Theta[:, ~small[:, k]]`). Stop as soon as the mask no longer changes.

</details>

<details><summary>Interpretation</summary>

SINDy finds the **form** of the equation (a short, readable formula), whereas a PINN fits the *parameters* of a known form. Noise hurts because finite differences divide by a small $\Delta t$: noise in $x$ becomes much larger noise in $\dot x$, and spurious terms start to fit it. What helps: smoothing before differentiating, more data, a library that is not too large, and a threshold above the small spurious coefficients but below the smallest true coefficient.

</details>

**Exercise (Slide 46):** read the application *“SINDy: Discovering the Dynamics of an Underwater Robot from Data”* (Markus Buchholz, Medium) or find another SINDy application yourself. Note: which state variables, which library, which threshold – and what went wrong or right?

## 14 · Equilibrium Points of Continuous-Time Models (Exercise 7.1 – Slides 51–54)

Same idea as in discrete time, but the left-hand side is now zero:
$$\frac{dx}{dt} = G(x) \quad\Rightarrow\quad 0 = G(x_{eq})$$

**Examples:** logistic growth → $x_{eq} = 0,\ K$.

**Exercise 7.1:** $\dfrac{dx}{dt} = x^2 - r x + 1$. Depending on $r$ there are two, one or zero equilibria.

**Tasks:**
1. Implement `equilibria_logistic(r, K)`.
2. Implement `equilibria_71(r)` returning the real equilibria as a sorted list (empty list if none).

In [ ]:
def equilibria_logistic(r, K):
    '''Equilibria of dx/dt = r x (1 - x/K) as a sorted list.'''
    # TODO: implement
    raise NotImplementedError


def equilibria_71(r, atol=1e-12):
    '''Real equilibria of dx/dt = x^2 - r x + 1 as a sorted list ([] if there are none).'''
    # TODO: implement
    raise NotImplementedError

In [ ]:
# Tests
assert equilibria_logistic(0.2, 5.0) == [0.0, 5.0]
assert np.allclose(equilibria_71(3.0), [(3 - np.sqrt(5)) / 2, (3 + np.sqrt(5)) / 2])
assert np.allclose(equilibria_71(-3.0), [(-3 - np.sqrt(5)) / 2, (-3 + np.sqrt(5)) / 2])
assert np.allclose(equilibria_71(2.0), [1.0]) and np.allclose(equilibria_71(-2.0), [-1.0])
assert equilibria_71(1.0) == [] and equilibria_71(0.0) == []
for r_ in [-5.0, -2.5, 2.5, 4.0]:
    for xe in equilibria_71(r_):
        assert abs(xe ** 2 - r_ * xe + 1) < 1e-9
print("✓ Ex. 7.1 equilibria")

<details><summary>Hint</summary>

Quadratic formula: $x_{eq} = \frac{r \pm \sqrt{r^2-4}}{2}$. The discriminant $r^2-4$ tells you how many real solutions exist: $>0$ two, $=0$ one ($x_{eq}=r/2$), $<0$ none.

</details>

<details><summary>Interpretation</summary>

At $|r|=2$ two equilibria merge and disappear when $|r|<2$ – a **saddle-node bifurcation**. Next section: how to tell which of the equilibria is stable.

</details>

## 15 · Phase Line, Stability and Basins of Attraction (Slides 56–66)

For $\dot x = G(x)$ the **sign of $G$** tells you which way $x$ moves: $G>0$ → right, $G<0$ → left, $G=0$ → equilibrium.

| Equilibrium | Flow left / right of it | Symbol |
|---|---|---|
| stable (attractor) | → · ← | filled dot |
| unstable (repellor) | ← · → | open dot |
| semistable | → · → or ← · ← | half-filled dot |

The derivative test: $G'(x_{eq})<0$ stable, $>0$ unstable, $=0$ **inconclusive** (look at the signs instead).

**Tasks:**
1. Implement `classify_1d(G, x_eq)` by looking at the sign of $G$ just left and right of the equilibrium.
2. Implement `stability_by_derivative(dG_at_eq)`.
3. Implement `limit_state(G, x0, equilibria)` – where does a trajectory starting at $x_0$ end up (`±np.inf` if it escapes)? This defines the **basin of attraction**.
4. Exercise: $\dot x = x^2$ – implement `blowup_time(x0)`.

In [ ]:
def classify_1d(G, x_eq, eps=1e-4):
    '''Returns "stable", "unstable" or "semistable" from the signs of G at x_eq - eps and x_eq + eps.'''
    # TODO: implement
    raise NotImplementedError


def stability_by_derivative(dG_at_eq, atol=1e-9):
    '''Returns "stable", "unstable" or "inconclusive" from G'(x_eq).'''
    # TODO: implement
    raise NotImplementedError


def limit_state(G, x0, equilibria):
    '''Follow the flow from x0: return the first equilibrium reached in the direction of G(x0), or +/-inf.'''
    # TODO: implement
    raise NotImplementedError


def blowup_time(x0):
    '''Finite blow-up time of dx/dt = x^2, x(0) = x0 (np.inf if the solution does not blow up).'''
    # TODO: implement
    raise NotImplementedError

In [ ]:
# Tests
r_, K_ = 0.7, 4.0
G_log = lambda x: r_ * x * (1 - x / K_)
assert classify_1d(G_log, 0.0) == "unstable" and classify_1d(G_log, K_) == "stable"

G_73 = lambda x: x ** 2 - 3 * x + 1                       # Slide 59
lo, hi = (3 - np.sqrt(5)) / 2, (3 + np.sqrt(5)) / 2
assert classify_1d(G_73, lo) == "stable" and classify_1d(G_73, hi) == "unstable"

assert classify_1d(lambda x: -x ** 3, 0.0) == "stable"      # Slide 60
assert classify_1d(lambda x: x ** 3, 0.0) == "unstable"
assert classify_1d(lambda x: x ** 2, 0.0) == "semistable"

assert [stability_by_derivative(v) for v in (-1.0, 2.0, 0.0)] == ["stable", "unstable", "inconclusive"]
assert stability_by_derivative(2 * 0.0) == "inconclusive", "G = x^2 has G'(0) = 0: the derivative test fails, signs are needed"

# Basins of attraction (Slide 61): stable points 1, 3, 5 / unstable points 2, 4
G_basin = lambda x: -(x - 1) * (x - 2) * (x - 3) * (x - 4) * (x - 5)
eqs = [1, 2, 3, 4, 5]
assert [classify_1d(G_basin, e) for e in eqs] == ["stable", "unstable", "stable", "unstable", "stable"]
for x0, lim in [(0.0, 1), (1.5, 1), (2.5, 3), (3.5, 3), (4.5, 5), (10.0, 5)]:
    assert limit_state(G_basin, x0, eqs) == lim, f"x0 = {x0} should end at {lim}"
assert limit_state(G_73, 3.0, [lo, hi]) == np.inf and limit_state(G_73, 1.0, [lo, hi]) == lo
assert limit_state(lambda x: x ** 2, -1.0, [0.0]) == 0.0 and limit_state(lambda x: x ** 2, 1.0, [0.0]) == np.inf

# dx/dt = x^2: x(t) = x0 / (1 - x0 t) blows up at t = 1/x0 (Slide 65)
assert np.isclose(blowup_time(0.5), 2.0) and blowup_time(-1.0) == np.inf
t, y = euler(lambda t, x: x ** 2, 0.5, 1.9, 1e-4)
assert y[-1] > 7, "Euler also sees the solution growing sharply close to t = 2"
print("✓ phase line, stability, basins")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, (G, name, eq_list, xl) in zip(axes, [(G_log, "logistic", [0.0, K_], (-1, 5.5)), (G_73, "x² − 3x + 1", [lo, hi], (-0.5, 3.5)),
                                              (G_basin, "5 equilibria (basins)", eqs, (0, 6))]):
    xs = np.linspace(*xl, 400)
    ax.plot(xs, G(xs), "k"); ax.axhline(0, color="gray", lw=0.8)
    for e in eq_list:
        c = classify_1d(G, e)
        ax.plot(e, 0, "o", ms=9, mfc={"stable": "k", "unstable": "w", "semistable": "0.5"}[c], mec="k", zorder=3)
    ax.set_title(f"G(x): {name}"); ax.set_xlabel("x"); ax.grid(alpha=0.3)
plt.tight_layout();

<details><summary>Hint 1</summary>

`np.sign(G(x_eq - eps))` and `np.sign(G(x_eq + eps))`. Left positive and right negative ⇒ trajectories on both sides move towards the point ⇒ stable. The reverse ⇒ unstable. The same sign on both sides ⇒ semistable.

</details>

<details><summary>Hint 2 (limit_state)</summary>

If `G(x0) > 0` the state moves right: the answer is the smallest equilibrium greater than `x0` (or `np.inf` if there is none). If `G(x0) < 0` it is the largest equilibrium smaller than `x0` (or `-np.inf`).

</details>

<details><summary>Hint 3 (blow-up)</summary>

Separate variables: $\int dx/x^2 = \int dt \Rightarrow -1/x = t + c \Rightarrow x(t) = \dfrac{x_0}{1 - x_0 t}$.

</details>

<details><summary>Interpretation (Slide 65)</summary>

$\dot x = x^2$ has a semistable point at 0: curves starting below rise towards 0, curves starting above rise away and blow up in **finite** time $1/x_0$. Here $G'(0)=0$, so the linearisation tells you nothing – that is exactly the "inconclusive" case.

</details>

## 16 · Phase Space: the Pendulum (Exercise 7.4 – Slides 67–69)

Continuous trajectories in phase space are smooth and never intersect (unlike in discrete time).

**Exercise 7.4:** draw the phase space of $\ \ddot\theta = -\frac gL\sin\theta\ \ $ ($L=g=1$). With $\omega = \dot\theta$:
$$\dot\theta = \omega,\qquad \dot\omega = -\frac gL\sin\theta$$

**Tasks:**
1. Implement `pendulum_rhs(t, s, g, L)`.
2. Implement `pendulum_energy(s)` ($E = \tfrac12\omega^2 - \tfrac gL\cos\theta$, conserved) and compare solvers.
3. Implement `jacobian_pendulum(theta, g, L)` and classify the equilibria $(0,0)$ and $(\pi,0)$ with `classify_equilibrium_2d` from Section 9.

In [ ]:
def pendulum_rhs(t, s, g=1.0, L=1.0):
    '''s = (theta, omega). Must also work when s is an array of shape (2, ...) (needed by the streamplot).'''
    # TODO: implement
    raise NotImplementedError


def pendulum_energy(s, g=1.0, L=1.0):
    '''E = 0.5 * omega^2 - (g/L) * cos(theta) for every row of s (shape (n, 2)).'''
    # TODO: implement
    raise NotImplementedError


def jacobian_pendulum(theta, g=1.0, L=1.0):
    '''2x2 Jacobian of the pendulum system at (theta, 0).'''
    # TODO: implement
    raise NotImplementedError

In [ ]:
# Tests
assert np.allclose(pendulum_rhs(0.0, [np.pi / 2, 0.3]), [0.3, -1.0])
assert np.allclose(pendulum_rhs(0.0, [0.0, 0.0]), [0, 0]) and np.allclose(pendulum_rhs(0.0, [np.pi, 0.0]), [0, 0], atol=1e-12)
assert np.isclose(pendulum_energy(np.array([[0.0, 0.0]]))[0], -1.0)

sol = solve_ivp(pendulum_rhs, [0, 20], [1.0, 0.0], rtol=1e-10, atol=1e-12, t_eval=np.linspace(0, 20, 2001))
drift_ivp = np.max(np.abs(pendulum_energy(sol.y.T) - pendulum_energy(sol.y.T)[0]))
t, y_e = euler(pendulum_rhs, [1.0, 0.0], 20.0, 0.01)
drift_eul = np.max(np.abs(pendulum_energy(y_e) - pendulum_energy(y_e)[0]))
print(f"energy drift: solve_ivp = {drift_ivp:.1e}, Euler = {drift_eul:.1e}")
assert drift_ivp < 1e-6 and drift_eul > 1e-2

assert np.allclose(jacobian_pendulum(0.0), [[0, 1], [-1, 0]])
assert classify_equilibrium_2d(jacobian_pendulum(0.0), atol=1e-9) == "center"
assert classify_equilibrium_2d(jacobian_pendulum(np.pi), atol=1e-9) == "saddle"
print("✓ Ex. 7.4 pendulum")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_vector_field(pendulum_rhs, (-3.2, 3.2), (-3, 3), ax=axes[0], title="Pendulum phase space (L = g = 1)", labels=(r"$\theta$", r"$\omega$"))
trajs = [solve_ivp(pendulum_rhs, [0, 30], s0, rtol=1e-9, t_eval=np.linspace(0, 30, 800)).y.T
         for s0 in [[0.5, 0], [1.5, 0], [2.5, 0], [-3.0, 1.0], [0, 2.2]]]
plot_phase(trajs, labels=(r"$\theta$", r"$\omega$"), title="Trajectories (closed = oscillation, open = rotation)", ax=axes[1])
axes[1].set_xlim(-8, 8)
plt.tight_layout();

<details><summary>Hint 1</summary>

`[s[1], -(g / L) * np.sin(s[0])]` – indexing `s[0]`, `s[1]` works both for a single state and for the grid arrays used by the streamplot.

</details>

<details><summary>Hint 2 (Jacobian)</summary>

$J = \begin{pmatrix} \partial\dot\theta/\partial\theta & \partial\dot\theta/\partial\omega\\ \partial\dot\omega/\partial\theta & \partial\dot\omega/\partial\omega\end{pmatrix} = \begin{pmatrix} 0 & 1\\ -\frac gL\cos\theta & 0\end{pmatrix}$.

</details>

<details><summary>Interpretation</summary>

At $(0,0)$ the eigenvalues are $\pm i\sqrt{g/L}$ – a center: small oscillations. At $(\pi,0)$ they are $\pm\sqrt{g/L}$ – a saddle: the inverted pendulum. Closed curves are oscillations, wavy open curves are full rotations. The energy $E$ is conserved by the true flow; Euler's energy drift (an outward spiral) is the same artifact as in Lotka–Volterra and in the linear center of Section 9.

</details>

## 17 · Variable Rescaling in Continuous Time (Slides 71–74)

In continuous time there is **one more variable to rescale: time**. For the logistic growth model
$$\frac{dx}{dt} = r x\left(1 - \frac xK\right)$$
substitute $x = \alpha x'$ and $t = \beta t'$:
$$\frac{dx'}{dt'} = r\beta\,x'\left(1 - \frac{\alpha x'}{K}\right) \xrightarrow{\ \alpha = K,\ \beta = 1/r\ } \frac{dx'}{dt'} = x'(1-x')$$

**No parameter left!** Unlike the discrete model, $r$ and $K$ only stretch time and population – they never change the qualitative behaviour.

**Tasks:**
1. Implement `rescale_params_logistic_ct(r, K)` returning $(\alpha, \beta)$ and the dimensionless right-hand side `nondim_rhs`.
2. Implement `half_time_nondim(x0p)` (time at which $x'=1/2$) and `physical_half_time(x0, r, K)`.
3. Verify the “one curve, stretched axes” picture (Slide 73): bacteria ($r=2$/h, $K=1000$) vs rabbits ($r=0.5$/year, $K=50$), both starting at 10 % of capacity.

In [ ]:
def rescale_params_logistic_ct(r, K):
    '''Returns (alpha, beta) for x = alpha x', t = beta t'.'''
    # TODO: implement
    raise NotImplementedError


def nondim_rhs(t, xp):
    '''dx'/dt' = x' (1 - x').'''
    # TODO: implement
    raise NotImplementedError


def half_time_nondim(x0p):
    '''Dimensionless time t' at which x' reaches 1/2, starting from x0p < 1/2.'''
    # TODO: implement (solve the exact logistic solution for x' = 1/2)
    raise NotImplementedError


def physical_half_time(x0, r, K):
    '''Physical time at which x reaches K/2.'''
    # TODO: implement (use the rescaling)
    raise NotImplementedError

In [ ]:
# Tests
assert rescale_params_logistic_ct(2.0, 1000.0) == (1000.0, 0.5)
assert np.isclose(nondim_rhs(0.0, 0.25), 0.1875)
assert np.isclose(half_time_nondim(0.1), np.log(9)) and np.isclose(half_time_nondim(0.1), 2.2, atol=0.01)

# A: bacteria, B: rabbits. Same dimensionless curve after rescaling x' = x/K, t' = r t
bact = dict(x0=100.0, r=2.0, K=1000.0)
rabb = dict(x0=5.0, r=0.5, K=50.0)
assert np.isclose(physical_half_time(**bact), 1.1, atol=0.01), "bacteria: half capacity after 1.1 h"
assert np.isclose(physical_half_time(**rabb), 4.4, atol=0.01), "rabbits: half capacity after 4.4 years"

sol = solve_ivp(nondim_rhs, [0, 7], [0.1], dense_output=True, rtol=1e-10, atol=1e-12)
for p in (bact, rabb):
    alpha, beta = rescale_params_logistic_ct(p["r"], p["K"])
    tt = np.linspace(0, 7 * beta, 30)
    x_scaled_back = alpha * sol.sol(tt / beta)[0]                       # x = alpha * x'(t / beta)
    assert np.allclose(x_scaled_back, logistic_exact(tt, p["x0"], p["r"], p["K"]), atol=1e-6), "rescaled curve must reproduce the original"
print("✓ continuous rescaling")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, p, name, unit in [(axes[0], bact, "A: bacteria", "hours"), (axes[1], rabb, "B: rabbits", "years")]:
    tt = np.linspace(0, 7 / p["r"], 300)
    ax.plot(tt, logistic_exact(tt, **p)); ax.axhline(p["K"], ls="--", c="gray")
    ax.set_title(f"{name}: r = {p['r']}, K = {p['K']:g}"); ax.set_xlabel(f"time ({unit})"); ax.grid(alpha=0.3)
    axes[2].plot(p["r"] * tt, logistic_exact(tt, **p) / p["K"], lw=3 if p is bact else 1.5, ls="-" if p is bact else "--")
axes[2].set_title("After rescaling: A and B on top of each other"); axes[2].set_xlabel("t' = r t"); axes[2].set_ylabel("x' = x / K"); axes[2].grid(alpha=0.3)
plt.tight_layout();

<details><summary>Hint 1</summary>

From $\dfrac{dx'}{dt'} = r\beta\,x'(1 - \alpha x'/K)$: the factor $\alpha/K$ must be $1$, and $r\beta$ must be $1$.

</details>

<details><summary>Hint 2 (half time)</summary>

Dimensionless solution: $x'(t') = \dfrac{1}{1 + \frac{1-x_0'}{x_0'}e^{-t'}}$. Set $x'=\tfrac12$ and solve for $t'$: $t' = \ln\dfrac{1-x_0'}{x_0'}$. Physical time: $t = \beta\,t'$.

</details>

<details><summary>Interpretation</summary>

$r$ stretches time (1.1 h versus 4.4 years), $K$ stretches the population (1000 versus 50), the shape never changes. Contrast with Section 8: the **discrete** logistic model keeps one parameter ($r'=1+r$) after rescaling, and that parameter controls period doubling and chaos. The continuous model has none – so any such behaviour in a simulation is a discretisation artifact, not a property of the model.

</details>

## Notes on the Slides & Big Picture Summary

| Concept | Discrete time (Week 3) | Continuous time (Week 4) |
|---|---|---|
| **Model** | $x_t = F(x_{t-1})$ | $\dot x = G(x)$ |
| **Simulation** | iterate $F$ | numerical integration (forward Euler: $x + G(x)\Delta t$) |
| **Equilibrium** | $x = F(x)$ | $G(x) = 0$ |
| **Linear stability** | $\lvert\lambda\rvert < 1$ | $\mathrm{Re}(\lambda) < 0$ |
| **1D stability** | $\lvert F'(x_{eq})\rvert < 1$ | $G'(x_{eq}) < 0$ |
| **Rescaling logistic** | one parameter left ($r'=1+r$) | no parameter left |

- **Euler artifacts:** Euler turns a center into an unstable spiral (Lotka–Volterra, pendulum) and, for large steps, creates period doubling and chaos in logistic growth ($r\Delta t > 2$). Always check that numerical results match the expected model behaviour.
- **Gradient descent is Euler** on the gradient flow: learning rate = time step, stable iff $\eta\lambda_{\max} < 2$.
- **PINNs** solve a known equation / learn its parameters via the residual; **SINDy** discovers the *form* of the equation by sparse regression.
- **Higher-order and non-autonomous** equations become first-order autonomous systems by adding state variables (velocity, clock).